<h2 style="text-align:center;">📝 Text Summarization using spaCy</h2>


### 📌 Introduction
- **Text Summarization** = process of reducing a long text into a shorter version while retaining its meaning.  
- Two main approaches:  
  1. **Extractive Summarization** → selects key sentences directly from the text.  
  2. **Abstractive Summarization** → generates new sentences (like humans summarize).  

👉 In this notebook, we’ll build a **basic extractive summarizer** using **spaCy**.  
It works by:  
1. Removing stopwords and punctuation.  
2. Calculating word frequencies.  
3. Scoring sentences based on important words.  
4. Selecting top-ranked sentences as the summary.  


In [1]:
# Install spaCy (only once per environment)
# !pip install spacy

# Download small English model
# !python -m spacy download en_core_web_sm

import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation


In [2]:
# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Example long text (you can replace this with any article or document)
text = """There are broadly two types of extractive summarization tasks depending on what the summarization program focuses on. 
The first is generic summarization, which focuses on obtaining a generic summary or abstract of the collection (whether documents, or sets of images, or videos, news stories etc.). 
The second is query relevant summarization, sometimes called query-based summarization, which summarizes objects specific to a query. 
Summarization systems are able to create both query relevant text summaries and generic machine-generated summaries depending on what the user needs.

An example of a summarization problem is document summarization, which attempts to automatically produce an abstract from a given document. 
Sometimes one might be interested in generating a summary from a single source document, while others can use multiple source documents (for example, a cluster of articles on the same topic). 
This problem is called multi-document summarization. A related application is summarizing news articles. 
Imagine a system, which automatically pulls together news articles on a given topic (from the web), and concisely represents the latest news as a summary."""


In [3]:
# Process the text with spaCy
doc = nlp(text)

# Tokens
tokens = [token.text for token in doc]
print("Number of tokens:", len(tokens))
print("First 20 tokens:", tokens[:20])


Number of tokens: 212
First 20 tokens: ['There', 'are', 'broadly', 'two', 'types', 'of', 'extractive', 'summarization', 'tasks', 'depending', 'on', 'what', 'the', 'summarization', 'program', 'focuses', 'on', '.', '\n', 'The']


In [4]:
# Stopwords list
stopwords = list(STOP_WORDS)
print("Number of stopwords in spaCy:", len(stopwords))
print("First 20 stopwords:", stopwords[:20])

# Punctuation list
print("Punctuation symbols:", punctuation)


Number of stopwords in spaCy: 326
First 20 stopwords: ['n’t', 'a', 'perhaps', 'often', 'show', 'just', 'nine', "'d", 'and', 'upon', 'across', 'hereby', 'by', 'none', 'someone', 'have', "'s", 'together', 'move', 'i']
Punctuation symbols: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [5]:
# Word frequency dictionary
word_frequencies = {}
for word in doc:
    if word.text.lower() not in stopwords and word.text.lower() not in punctuation:
        if word.text.lower() not in word_frequencies.keys():
            word_frequencies[word.text.lower()] = 1
        else:
            word_frequencies[word.text.lower()] += 1

print("Top 10 word frequencies:")
print(dict(list(word_frequencies.items())[:10]))


Top 10 word frequencies:
{'broadly': 1, 'types': 1, 'extractive': 1, 'summarization': 9, 'tasks': 1, 'depending': 2, 'program': 1, 'focuses': 2, '\n': 6, 'generic': 3}


In [6]:
# Normalize frequencies (divide by max frequency)
max_frequency = max(word_frequencies.values())

for word in word_frequencies.keys():
    word_frequencies[word] = word_frequencies[word] / max_frequency

print("Top 10 normalized frequencies:")
print(dict(list(word_frequencies.items())[:10]))


Top 10 normalized frequencies:
{'broadly': 0.1111111111111111, 'types': 0.1111111111111111, 'extractive': 0.1111111111111111, 'summarization': 1.0, 'tasks': 0.1111111111111111, 'depending': 0.2222222222222222, 'program': 0.1111111111111111, 'focuses': 0.2222222222222222, '\n': 0.6666666666666666, 'generic': 0.3333333333333333}


In [7]:
# Get sentences
sentence_tokens = [sent for sent in doc.sents]
print("Number of sentences:", len(sentence_tokens))

# Score each sentence
sentence_scores = {}
for sent in sentence_tokens:
    for word in sent:
        if word.text.lower() in word_frequencies.keys():
            if sent not in sentence_scores.keys():
                sentence_scores[sent] = word_frequencies[word.text.lower()]
            else:
                sentence_scores[sent] += word_frequencies[word.text.lower()]

print("Sample sentence scores:")
list(sentence_scores.items())[:3]


Number of sentences: 9
Sample sentence scores:


[(There are broadly two types of extractive summarization tasks depending on what the summarization program focuses on. ,
  3.6666666666666665),
 (The first is generic summarization, which focuses on obtaining a generic summary or abstract of the collection (whether documents, or sets of images, or videos, news stories etc.). ,
  4.555555555555556),
 (The second is query relevant summarization, sometimes called query-based summarization, which summarizes objects specific to a query. ,
  5.000000000000001)]

#### Select Top Sentences

In [8]:
from heapq import nlargest

# Select top 30-40% sentences
select_length = int(len(sentence_tokens) * 0.4)

# Get top sentences
summary_sentences = nlargest(select_length, sentence_scores, key=sentence_scores.get)

# Combine into final summary
final_summary = " ".join([sent.text for sent in summary_sentences])

print("📌 Final Extractive Summary:\n")
print(final_summary)


📌 Final Extractive Summary:

The second is query relevant summarization, sometimes called query-based summarization, which summarizes objects specific to a query. 
 An example of a summarization problem is document summarization, which attempts to automatically produce an abstract from a given document. 
 The first is generic summarization, which focuses on obtaining a generic summary or abstract of the collection (whether documents, or sets of images, or videos, news stories etc.). 



<h2 style="text-align:center;">✅ Summary</h2>

- We implemented **Extractive Summarization** using **spaCy**.  
- Steps followed:  
  1. Tokenize text with spaCy.  
  2. Remove stopwords & punctuation.  
  3. Calculate word frequencies.  
  4. Score sentences based on important words.  
  5. Extract top sentences to form summary.  

### 🔹 Pros
- Simple, fast, and unsupervised (no training needed).  
- Works on any text.  

### 🔹 Cons
- Extractive only → does not generate new sentences.  
- May miss context or coherence.  
- Not as powerful as modern **transformer-based abstractive summarizers (BERT, GPT, etc.)**.  

🚀 Next step: Compare extractive methods with **transformer-based summarization** using Hugging Face.  
